In [266]:
import pandas as pd
import numpy as np
import io
import re
from IPython.display import display, HTML
from typing import Dict, List
import camelot
import os
import warnings
warnings.filterwarnings('ignore')

# Selenium imports
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
import time

# --- CONFIGURATION & LOCATORS ---
GameNumber = '6512971' #
URL = "https://www.ncaa.com/game/"+GameNumber+"/boxscore"

# Team Names (Used for PBP filtering and substitutions)
team_name = "Wis.-Whitewater"
opponent_name = "Elmhurst"

team_homeoraway = 'home'
usepdf = 'no'


# Substitution Strings (Normalized to lowercase 'in'/'out')
TEAM_SUB_OUT_STRING_NEW = f"Subbing out for {team_name}-"
TEAM_SUB_IN_STRING_NEW = f"Subbing in for {team_name}-"
OPPONENT_SUB_OUT_STRING_NEW = f"Subbing out for {opponent_name}-"
OPPONENT_SUB_IN_STRING_NEW = f"Subbing in for {opponent_name}-"

# Boxscore Locators
TEAM_SELECTOR_CLASS = "boxscore-team-selector-team homeTeam-bg-primary_color awayTeam-border-primary_color home"

# "boxscore-team-selector-team awayTeam-bg-primary_color homeTeam-border-primary_color away active"

TABLE_CONTAINER_CLASS = "boxscore-table-collection"
PLAY_BY_PLAY_HREF = "/game/"+GameNumber+"/play-by-play"
PLAY_BY_PLAY_TABLE_CONTAINER = "gamecenter-tab-play-by-play"

# --- HELPER FUNCTIONS ---

# def extract_player_name(play,acting_team, usepdf):
#     """Extracts the player name, specifically for substitution records."""
#     # if TEAM_SUB_OUT_STRING_NEW in play:
#     #     return play.split(TEAM_SUB_OUT_STRING_NEW)[-1].strip()
#     # elif TEAM_SUB_IN_STRING_NEW in play:
#     #     return play.split(TEAM_SUB_IN_STRING_NEW)[-1].strip()
#     # elif OPPONENT_SUB_OUT_STRING_NEW in play:
#     #     return play.split(OPPONENT_SUB_OUT_STRING_NEW)[-1].strip()
#     # elif OPPONENT_SUB_IN_STRING_NEW in play:
#     #     return play.split(OPPONENT_SUB_IN_STRING_NEW)[-1].strip()
#     # print(usepdf)
#     if usepdf == 'no':
#         # team_suffix = acting_team + "'s"
#         if 'Subbing' in play:
#             play = play.split('-')[-1]
#             player_name = play.split(' ', 2)[0] + ' ' + play.split(' ', 2)[1]
#         else:
#             player_name = play.split(acting_team)[-1].strip()
#             if ' ' in player_name:
#                 player_name = player_name.split(' ', 2)[0] + ' ' + player_name.split(' ', 2)[1]
        
#     elif usepdf == 'yes':
#         player_name = play.split(' ', 2)[0] + ' ' + play.split(' ', 2)[1]

#     return player_name

# def get_player_info(play, player, actionby, team_key, team_name):
#     """
#     Extracts player name and action ('OUT' or 'IN') for a specific team.
#     Returns (action, player_name) or (None, None).
#     """
    
    # # Team-specific strings (replace with your actual team names)
    # sub_out_str_long = f"Subbing out for {team_name}-"
    # sub_in_str_long = f"Subbing in for {team_name}-" # ASSUMING 'Subbing in' uses the same format
    
    # # 1. Check for the specific 'Subbing out/in' format
    # if sub_out_str_long in play:
    #     try:
    #         player_name = play.split(sub_out_str_long)[1].split(" ")[0].strip()
    #         player_name = 
    #         return ('OUT', player_name)
    #     except IndexError:
    #         pass
            
    # if sub_in_str_long in play:
    #     try:
    #         player_name = play.split(sub_in_str_long)[1].split(" ")[0].strip()
    #         return ('IN', player_name)
    #     except IndexError:
    #         pass


    # # If the play contains 'Subs Out' AND the player is on the current team's court
    # # You would need a separate helper to find the player before 'Subs Out'.
    # if "Subs Out" in play and team_name == actionby:
    #     player_name = play.split(' ', 2)[0] + ' ' + play.split(' ', 2)[1]
    #      # Placeholder for extracting player name before "Subs Out"
    #     return ('OUT', player_name)

    # if "Subs In" in play and team_name == actionby:
    #     player_name = play.split(' ', 2)[0] + ' ' + play.split(' ', 2)[1]
    #      # Placeholder for extracting player name before "Subs In"
    #     return ('IN', player_name)

    # return (None, None)

def time_to_seconds(time_str):
    """Converts MM:SS time string to total seconds."""
    if pd.isna(time_str) or not isinstance(time_str, str):
        return 0
    try:
        m, s = map(int, time_str.split(':'))
        return m * 60 + s
    except ValueError:
        return 0

def seconds_to_time(seconds):
    """Converts total seconds back to MM:SS format."""
    seconds = int(round(seconds))
    minutes = seconds // 60
    seconds = seconds % 60
    return f"{minutes:02d}:{seconds:02d}"

def determine_eop(play, TEAM_points, OPPONENT_points):
    """Determines if a play marks the end of a possession."""
    play = str(play).lower()
    
    if TEAM_points > 0 or OPPONENT_points > 0:
        return True
    if 'turnover by' in play:
        return True
    if 'offensive foul by' in play:
        return True
    if 'end of' in play and 'half' in play:
        return True
    
    return False

def lineup_html_formatter(lineup_list):
    """Converts the list of players into an HTML string with <br> breaks."""
    if isinstance(lineup_list, list):
        return '<br>'.join(lineup_list)
    return str(lineup_list)


# --- REFACTORED UNIFIED HELPER FUNCTIONS ---

def get_starters_list(df_pbp, sub_out_str, sub_in_str, usepdf, team_name):
    """
    Determines starters for a given team based on first-half substitution patterns.
    (Replaces TEAM/OPPONENT Starter Logic duplication)
    """
    ##################################


    if usepdf == 'no':
        print('Not using PDF')
        # Filter 'Sub Out' plays in the first half (HALF==1)
        filtered_subs_out = df_pbp[['PLAY','PLAYER','HALF','TIME']][
            (df_pbp['PLAY'].str.contains(sub_out_str, na=False)) & (df_pbp['HALF']==1)
        ].copy()
        for index, row in filtered_subs_out.iterrows():
            play = row['PLAY']
            # filtered_subs_out.loc[index, 'PLAYER'] = extract_player_name(play,team_name,usepdf)
        filtered_subs_out = filtered_subs_out.rename(columns={'TIME': 'SUB_OUT'}).drop(columns=['PLAY']).drop_duplicates(
            subset=['HALF', 'PLAYER'], keep='first'
        )
    
        # Filter 'Sub In' plays
        filtered_subs_in = df_pbp[['PLAY','PLAYER','HALF','TIME']][
            df_pbp['PLAY'].str.contains(sub_in_str, na=False)
        ].copy()
        for index, row in filtered_subs_in.iterrows():
            play = row['PLAY']
            # filtered_subs_in.loc[index, 'PLAYER'] = extract_player_name(play,team_name,usepdf)
        filtered_subs_in = filtered_subs_in.rename(columns={'TIME': 'SUB_IN'}).drop(columns=['PLAY']).drop_duplicates(
            subset=['HALF', 'PLAYER'], keep='first'
        )
    else:
        print('Using PDF')
        # # print(sub_in_str)
        # # print(sub_in_str.split('for ')[1])
        # if team_name in sub_in_str.split('for ')[1]:
        #     team_name = team_name
        #     print('Using team_name')
        # else:
        #     team_name = opponent_name
        #     print('Using opponent')

        print('splitting '+team_name)
            
        # Filter 'Sub Out' plays in the first half (HALF==1)
        filtered_subs_out = df_pbp[['PLAY','PLAYER','HALF','TIME']][
            (df_pbp['PLAY'].str.contains('Subs Out', na=False)) & (df_pbp['HALF']==1) & (df_pbp['ACTING_TEAM']==team_name)
        ].copy()
        for index, row in filtered_subs_out.iterrows():
            play = row['PLAY']
            # filtered_subs_out.loc[index, 'PLAYER'] = extract_player_name(play,team_name,usepdf)
        filtered_subs_out = filtered_subs_out.rename(columns={'TIME': 'SUB_OUT'}).drop(columns=['PLAY']).drop_duplicates(
            subset=['HALF', 'PLAYER'], keep='first'
        )
        
        # Filter 'Sub In' plays
        filtered_subs_in = df_pbp[['PLAY','PLAYER','HALF','TIME']][
            (df_pbp['PLAY'].str.contains('Subs In', na=False)) & (df_pbp['HALF']==1) & (df_pbp['ACTING_TEAM']==team_name)
        ].copy()
        filtered_subs_in['PLAYER'] = filtered_subs_in['PLAY'].str.split(" Subs",expand=True)[0]
        for index, row in filtered_subs_in.iterrows():
            play = row['PLAY']
            # filtered_subs_in.loc[index, 'PLAYER'] = extract_player_name(play,team_name,usepdf)
        filtered_subs_in = filtered_subs_in.rename(columns={'TIME': 'SUB_IN'}).drop(columns=['PLAY']).drop_duplicates(
            subset=['HALF', 'PLAYER'], keep='first'
        )

    
    # Merge to compare in/out times
    filtered_subs = pd.merge(filtered_subs_out, filtered_subs_in, how='left', on=['HALF','PLAYER'])
    
    # Convert times to seconds for comparison
    filtered_subs['SUB_OUT_SEC'] = filtered_subs['SUB_OUT'].apply(time_to_seconds)
    filtered_subs['SUB_IN_SEC'] = filtered_subs['SUB_IN'].apply(time_to_seconds)
    
    # Starter Logic: Sub out time > Sub in time, or sub in is missing (only subbed out in 1st half)
    filtered_subs['Starter'] = np.where(
        (filtered_subs['SUB_OUT_SEC'] > filtered_subs['SUB_IN_SEC']) | (filtered_subs['SUB_IN'].isna()), 
        'Yes', 
        'No'
    )
    
    return filtered_subs[filtered_subs['Starter']=='Yes']['PLAYER'].sort_values().to_list()


def calculate_lineup_ratings(df_pbp, team_name, opponent_name, agg_dict):
    """
    Performs line-up aggregation and advanced rating calculations for a single team.
    (Replaces TEAM/OPPONENT Lineup Aggregation duplication)
    """
    team_prefix = 'TEAM' if team_name == team_name else 'OPPONENT'
    opp_prefix = 'OPPONENT' if team_name == team_name else 'TEAM'
    
    roster_col = f'{team_prefix}_ROSTER_ON_COURT'
    points_for_col = f'{team_prefix}_POINTS'
    points_against_col = f'{opp_prefix}_POINTS'

    # 1. Aggregate Line-ups
    aggregated_data = df_pbp.groupby(roster_col).agg(agg_dict).reset_index()
    aggregated_data[roster_col] = aggregated_data[roster_col].str.split(',')
    aggregated_data['player_length'] = aggregated_data[roster_col].apply(len)
    aggregated_data = aggregated_data[aggregated_data['player_length'] == 5].sort_values(by='DURATION_SECONDS', ascending=False)
    aggregated_data = aggregated_data.rename(columns={'IS_END_OF_POSSESSION': 'POSSESSIONS'})
    
    # 2. Ratings calculations
    aggregated_data['Points For Per 40 Mins'] = (aggregated_data[points_for_col] / aggregated_data['DURATION_SECONDS']) * 2400
    aggregated_data['Points Against Per 40 Mins'] = (aggregated_data[points_against_col] / aggregated_data['DURATION_SECONDS']) * 2400
    aggregated_data['Offensive Rating'] = (aggregated_data[points_for_col] / aggregated_data['POSSESSIONS']) * 100
    aggregated_data['Defensive Rating'] = (aggregated_data[points_against_col] / aggregated_data['POSSESSIONS']) * 100
    aggregated_data['Net Rating'] = aggregated_data['Offensive Rating'] - aggregated_data['Defensive Rating']

    # 3. ROUNDING & NAN/INF HANDLING
    rounding_cols = ['Points For Per 40 Mins', 'Points Against Per 40 Mins',
                     'Offensive Rating', 'Defensive Rating', 'Net Rating',
                     points_for_col, points_against_col]
                         
    aggregated_data[rounding_cols] = aggregated_data[rounding_cols].fillna(0).replace([np.inf, -np.inf], 0)
    aggregated_data[rounding_cols] = aggregated_data[rounding_cols].round(0).astype(int)

    # 4. Final Formatting
    aggregated_data['AGGREGATED_TIME_MM:SS'] = aggregated_data['DURATION_SECONDS'].apply(seconds_to_time)
    final_results = aggregated_data.drop(columns=['DURATION_SECONDS', 'player_length']).rename(columns={
        roster_col: 'LINEUP', points_for_col: 'Points For', points_against_col: 'Points Against'
    })
    final_results['Plus/Minus'] = final_results['Points For'] - final_results['Points Against']
    
    final_results = final_results[['LINEUP', 'AGGREGATED_TIME_MM:SS', 'POSSESSIONS', 'Points For', 'Points Against', 
                                   'Plus/Minus', 'Offensive Rating', 'Defensive Rating', 'Net Rating', 
                                   'Points For Per 40 Mins', 'Points Against Per 40 Mins']]

    return final_results

def calculate_roster_traits(roster_str: str, df_class: pd.DataFrame) -> Dict[str, int]:
    """Calculates the count of each player trait for a given roster string."""
    roster_str = roster_str.replace('nan','None')
    target_players = [name.strip() for name in roster_str.split(',')] 
    
    # 1. Filter scouted players and add unscouted players
    df_scouted = df_class[df_class['TEAM/PLAYER'].isin(target_players)]
    scouted_names = df_scouted['TEAM/PLAYER'].tolist()
    not_scouted_names = [name for name in target_players if name not in scouted_names]
    
    df_not_scouted = pd.DataFrame({'Player Type': ['Not Scouted'] * len(not_scouted_names)})
    
    # 2. Combine and explode traits
    all_types = pd.concat([df_scouted['Player Type'], df_not_scouted['Player Type']])
    all_traits = (
        all_types
        .str.split(',\s*')
        .explode()
        .str.strip()
    )
    
    # 3. Count
    trait_counts = all_traits.value_counts()
    
    return trait_counts.to_dict()
    
def add_roster_traits_to_dataframe(df_pbp: pd.DataFrame, df_class: pd.DataFrame, roster_col: str) -> pd.DataFrame:
    """
    Applies the trait calculation efficiently and concatenates the resulting columns.
    This replaces the need for the slow 'for index, row in ... iterrows():' loop.
    """
    
    # Calculate traits for each row, resulting in a Series of dictionaries
    trait_dicts = df_pbp[roster_col].apply(lambda x: calculate_roster_traits(x, df_class))
    
    # Convert the Series of dictionaries into a new DataFrame (df_traits)
    df_traits = pd.DataFrame(trait_dicts.tolist())
    
    # Fill any NaNs (where a trait didn't exist in a row) with 0
    df_traits = df_traits.fillna(0).astype(int)
    
    # Concatenate the new trait columns (df_traits) with the original DataFrame
    df_result = pd.concat([df_pbp, df_traits], axis=1)
    
    return df_result
    
# --- CORE FUNCTION: ENRICH PBP DF WITH STATS (Finalized Logic with AST and STL) ---
def add_player_stats_to_pbp(df_pbp, team_name, opponent_name, usepdf):
    """
    Adds PLAYER, ACTING_TEAM, and unified stat columns (PTS, FGM, etc.) 
    to the PBP DataFrame, now correctly handling 'Assist by' and 'Steal by' as standalone events.
    """
    df = df_pbp.copy()
    TEAM_suffix = team_name + "'s"
    OPPONENT_suffix = opponent_name + "'s"
    
    
    if usepdf == 'no':
        df['ACTING_TEAM'] = None
 
        
    for stat in ['PTS', 'FGM', 'FGA', '3PM', '3PA', 'FTM', 'FTA', 'FOULS', 'TO', 'AST', 'STL', 'OREB', 'DREB']:
        df[stat] = 0
    
    missed_3pt_patterns = 'Missed 3-pointer|Missed Jumper|Jumper Missed|MISSED Jumper|Jumper MISSED'
    missed_2pt_patterns = 'Missed Layup|Missed Dunk|Missed Hook|Layup Missed|Dunk Missed|Hook Missed|Layup MISSED|Dunk MISSED|Hook MISSED'

    # 1. Identify all players (remains the same)
    all_players = set()
    for col in ['TEAM_ROSTER_ON_COURT', 'OPPONENT_ROSTER_ON_COURT']:
        for roster in df[col].dropna():
            all_players.update(roster.split(','))
    
    # 2. Iterate through rows to assign player and stats
    for index, row in df.iterrows():
        play = row['PLAY']
        acting_player = None
        acting_team = None
        points_col = None
        # player_name = None

        # # Get current rosters (remains the same)
        # TEAM_roster = set(row['TEAM_ROSTER_ON_COURT'].split(',')) if row['TEAM_ROSTER_ON_COURT'] and isinstance(row['TEAM_ROSTER_ON_COURT'], str) else set()
        # OPPONENT_roster = set(row['OPPONENT_ROSTER_ON_COURT'].split(',')) if row['OPPONENT_ROSTER_ON_COURT'] and isinstance(row['OPPONENT_ROSTER_ON_COURT'], str) else set()

        # print(TEAM_roster)
        
        # --- A. Check for Explicit Plays (Scoring/Attempt/Rebound/Assist/Steal - NEW STL LOGIC) ---
        if TEAM_suffix in play:
            # print('Teamsuffix')
            acting_team = team_name
            # player_name = extract_player_name(play,acting_team, usepdf)
                    
        elif OPPONENT_suffix in play:
            acting_team = opponent_name
            # player_name = extract_player_name(play,acting_team, usepdf)



        #######################
        elif usepdf == 'yes':
            acting_team = row['ACTING_TEAM']
            # player_name = row['PLAY'].split(' ', 2)[0] + ' ' + row['PLAY'].split(' ', 2)[1]
            
        ########################

        if row['TEAM_POINTS'] > row['OPPONENT_POINTS']:
                points_col = 'TEAM_POINTS'
        else:
            points_col = 'OPPONENT_POINTS'

        # df.loc[index, 'PLAYER'] = player_name
        df.loc[index, 'ACTING_TEAM'] = acting_team
                
        # # 1. Check for Assists
        if play.startswith('Assist by'):
            df.loc[index, 'AST'] = 1
            continue # Skip the rest of the stat assignment logic for this row

        elif 'Assists' in play and usepdf=='yes':
            df.loc[index, 'AST'] = 1

        # 2. Check for Steals (NEW LOGIC)
        elif play.startswith('Steal by'):
            df.loc[index, 'STL'] = 1
            continue # Skip the rest of the stat assignment logic for this row

        elif "Steal" in play and usepdf=='yes':
            df.loc[index, 'STL'] = 1

        # # 3. TEAM Team Play (Scoring/Attempt/Rebound)
        # elif TEAM_suffix in play:
        #     acting_team = team_name
        #     points_col = 'TEAM_POINTS'
        #     suffix_idx = play.find(TEAM_suffix) + len(TEAM_suffix)
            
        #     remaining_play = play[suffix_idx:].strip()
        #     player_candidates = sorted([p for p in TEAM_roster if p in remaining_play], key=len, reverse=True)

        #     if player_candidates:
        #         for player in player_candidates:
        #             if remaining_play.startswith(player):
        #                 acting_player = player
        #                 break
                        
        # # 4. OPPONENT Team Play (Scoring/Attempt/Rebound)
        # elif OPPONENT_suffix in play:
        #     acting_team = opponent_name
        #     points_col = 'OPPONENT_POINTS'
        #     suffix_idx = play.find(OPPONENT_suffix) + len(OPPONENT_suffix)

        #     remaining_play = play[suffix_idx:].strip()
        #     player_candidates = sorted([p for p in OPPONENT_roster if p in remaining_play], key=len, reverse=True)

        #     if player_candidates:
        #         for player in player_candidates:
        #             if remaining_play.startswith(player):
        #                 acting_player = player
        #                 break

        # # --- B. Check for Fouls and Turnovers (Secondary Action - Same as before) ---
        # if not acting_player:
        #     player_list = [p for p in all_players if p in play]
        #     # ... (Foul and Turnover logic remains unchanged)

        #     for player in sorted(player_list, key=len, reverse=True): 
                
        #         # Check for any kind of Foul
        #         if (("Foul on" in play or "Personal Foul on" in play) and player in play) or ("Offensive foul by" in play and player in play):
        #             acting_player = player
        #             if player in TEAM_roster:
        #                 acting_team = team_name
        #                 points_col = 'TEAM_POINTS'
        #             elif player in OPPONENT_roster:
        #                 acting_team = opponent_name
        #                 points_col = 'OPPONENT_POINTS'
        #             break
                    
        #         # Check for Turnover (Non-Foul)
        #         elif "Turnover by" in play and player in play:
        #             acting_player = player
        #             if player in TEAM_roster:
        #                 acting_team = team_name
        #                 points_col = 'TEAM_POINTS'
        #             elif player in OPPONENT_roster:
        #                 acting_team = opponent_name
        #                 points_col = 'OPPONENT_POINTS'
        #             break

        # if not acting_player or not acting_team:
        #     print('No acting player or team')
        #     continue
            
        # Assign Player and Team to Row (for primary action)
        # df.loc[index, 'PLAYER'] = acting_player
        df.loc[index, 'ACTING_TEAM'] = acting_team
        
        # --- C. Stat Allocation (Remains the same for all non-Assist/non-Steal plays) ---
        
        # 1. Offensive Fouls (FOULS = 1, TO = 1)
        if "Offensive foul by" in play:
            df.loc[index, ['FOULS', 'TO']] = 1
        
        # 2. Defensive/Standard Fouls (FOULS = 1 only)
        elif "Foul on" in play: 
            df.loc[index, 'FOULS'] = 1

        elif "Commits Personal Foul" in play and usepdf=='yes':
            df.loc[index, 'FOULS'] = 1
        
        # 3. Non-Foul Turnovers (TO = 1 only)
        elif "Turnover by" in play: 
            df.loc[index, 'TO'] = 1

        elif "Turnover" in play and usepdf=='yes':
            df.loc[index, 'TO'] = 1
            
        # 4. Rebounds
        if 'Offensive rebound' in play:
            df.loc[index, 'OREB'] = 1
        elif 'Offensive Rebound' in play and usepdf=='yes':
            df.loc[index, 'OREB'] = 1

        elif 'Defensive rebound' in play:
            df.loc[index, 'DREB'] = 1
        elif 'Defensive Rebound' in play and usepdf=='yes':
            df.loc[index, 'DREB'] = 1

        # Scoring Plays 
        if row[points_col] > 0:
            # ... (Scoring logic remains unchanged)
            df.loc[index, 'PTS'] = row[points_col]
            
            if 'Free Throw by' in play:
                df.loc[index, ['FTM', 'FTA']] = 1
            elif 'Makes Free Throw' in play and usepdf=='yes':
                df.loc[index, ['FTM', 'FTA']] = 1
            
            elif row[points_col] == 3:
                df.loc[index, ['3PM', '3PA', 'FGM', 'FGA']] = 1
            elif row[points_col] == 2:
                df.loc[index, ['FGM', 'FGA']] = 1
        
        # Missed Shots (Points == 0)
        elif row[points_col] == 0: 
            # ... (Missed shot logic remains unchanged)
            if 'Free Throw MISSED' in play:
                df.loc[index, 'FTA'] = 1
            elif 'Misses Free Throw' in play and usepdf=='yes':
                df.loc[index, 'FTA'] = 1
            elif any(m_pat in play for m_pat in missed_3pt_patterns.split('|')):
                if not df.loc[index, 'FTA']: 
                    df.loc[index, ['3PA', 'FGA']] = 1
            elif any(m_pat in play for m_pat in missed_2pt_patterns.split('|')):
                if not df.loc[index, 'FTA'] and not df.loc[index, '3PA']:
                     df.loc[index, 'FGA'] = 1
            elif "Misses 2PT" in play and usepdf=='yes':
                df.loc[index, 'FGA'] = 1
            elif "Misses 3PT" in play and usepdf=='yes':
                df.loc[index, ['3PA', 'FGA']] = 1
        
    return df






# --- BOX SCORE CREATION FUNCTION ---
def create_box_score(df_pbp_enriched, team_name):
    """
    Generates a player-based box score by filtering the enriched df_pbp 
    by ACTING_TEAM and aggregating, and calculating percentages. (AST added)
    """
    
    # Filter for plays where this team was the actor
    df_team = df_pbp_enriched[df_pbp_enriched['ACTING_TEAM'] == team_name].copy()
    
    # Aggregate stats directly (AST included)
    stats_to_sum = ['PTS', 'FGM', 'FGA', '3PM', '3PA', 'FTM', 'FTA', 'FOULS', 'TO', 'AST', 'STL', 'OREB', 'DREB'] 
    box_score = df_team.groupby('PLAYER')[stats_to_sum].sum().astype(int).reset_index()

    # Calculate Total Rebounds
    box_score['REB'] = box_score['OREB'] + box_score['DREB']

    # --- Final Formatting (Calculating Percentages) ---
    box_score['FG%'] = np.where(box_score['FGA'] > 0, (box_score['FGM'] / box_score['FGA']), 0).round(3)
    box_score['3P%'] = np.where(box_score['3PA'] > 0, (box_score['3PM'] / box_score['3PA']), 0).round(3)
    box_score['FT%'] = np.where(box_score['FTA'] > 0, (box_score['FTM'] / box_score['FTA']), 0).round(3)
    
    # Reorder for presentation (AST included)
    box_score = box_score[[
        'PLAYER', 'PTS', 'FGM', 'FGA', 'FG%', '3PM', '3PA', '3P%', 'FTM', 'FTA', 'FT%', 
        'OREB', 'DREB', 'REB', 'AST', 'STL', 'TO', 'FOULS'
    ]]
    
    # Add a totals row
    totals = box_score[['PTS', 'FGM', 'FGA', '3PM', '3PA', 'FTM', 'FTA', 'AST', 'STL', 'OREB', 'DREB', 'REB', 'TO', 'FOULS']].sum()
    totals['PLAYER'] = 'TOTAL'
    
    totals['FG%'] = round((totals['FGM'] / totals['FGA']), 3) if totals['FGA'] > 0 else 0
    totals['3P%'] = round((totals['3PM'] / totals['3PA']), 3) if totals['3PA'] > 0 else 0
    totals['FT%'] = round((totals['FTM'] / totals['FTA']), 3) if totals['FTA'] > 0 else 0
    
    # Ensure totals are in the correct order before assigning
    totals = totals.reindex(box_score.columns).fillna('')
    box_score.loc['TOTAL'] = totals
    
    return box_score.fillna('')

import pandas as pd
import numpy as np
import re

# NOTE: You MUST ensure 'team_name' and 'opponent_name' passed to the 
# main function match the column names in the PDF (e.g., 'UW-WHITEWATER' and 'EUREKA').

def clean_pbp_pdf_format(df_pbp, team_col, opponent_col):
    """
    Cleans and standardizes the Play-by-Play DataFrame specifically
    when it comes from the 'PDF-style' scraped HTML.
    
    This merges the two team-specific play columns into a single 'PLAY' column.
    """
    print(f"\n--- ⚠️ PDF-Style Data Cleaning Engaged: Merging '{team_col}' and '{opponent_col}' plays ---")

    # 1. Merge the two team columns into a single 'PLAY' column
    # Use pandas 'combine_first' or similar logic. numpy.where is reliable here.
    # The columns are already in UPPERCASE due to a line in your main script.
    df_pbp['PLAY'] = np.where(
        df_pbp[team_col].notna(), 
        df_pbp[team_col].astype(str).str.strip(), 
        df_pbp[opponent_col].astype(str).str.strip()
    )
    
    # Drop the original split columns
    df_pbp.drop(columns=[team_col, opponent_col], inplace=True, errors='ignore')
    
    # 2. Cleanup artifacts from PDF processing
    # The PDF example shows rows with 'Start 1st Half' that might not have a TIME, 
    # and the score format seems to be 'TS-OS' not '(TS-OS)' (which your score parsing expects).
    
    # Remove rows that are empty placeholders
    df_pbp.replace('nan', np.nan, inplace=True)
    df_pbp.dropna(subset=['PLAY'], inplace=True)
    
    # The 'TIME' column contains '(H1)' or '(H2)'. We need to isolate time and clean the play.
    # Your main script already handles time calculation later, but we should clean the 
    # 'TIME' and 'PLAY' columns of the 'Half' indicator if necessary, although 
    # your main script seems to manage the HALF column separately.
    
    # Let's ensure 'SCORE' is in the format expected by the main script's score parser: (T-O)
    # The PDF shows '2-0', '4-0', '4-3'[cite: 85]. We need to wrap it.
    df_pbp['SCORE'] = df_pbp['SCORE'].apply(lambda x: f'({x})' if re.match(r'^\d+-\d+$', str(x)) else x)
    
    # Reorder columns to ensure consistency for debugging/future steps
    if 'TIME' in df_pbp.columns and 'SCORE' in df_pbp.columns:
        df_pbp = df_pbp[['TIME', 'PLAY', 'SCORE'] + [col for col in df_pbp.columns if col not in ['TIME', 'PLAY', 'SCORE']]]
    
    print("--- Cleaning successful. Standard columns ('TIME', 'PLAY', 'SCORE') created. ---")

    return df_pbp


def scrape_pdf_to_dataframe(pdf_file_name,game,date,team,opponent):
    """
    Scrapes tables from a PDF file using camelot-py (lattice flavor).
    """

    
    
    if not os.path.exists(pdf_file_name):
        print(f"Error: File not found at '{pdf_file_name}'.")
        return None

    print(f"Attempting to extract tables from: {pdf_file_name} using camelot...")

    try:
        # Use 'lattice' flavor for grid-like data like a play-by-play log
        tables = camelot.read_pdf(
            pdf_file_name,
            pages="all",
            flavor='stream',
            suppress_stdout=True
        )

        if not tables:
            print("No tables were detected by camelot-py.")
            return []

        print(f"Successfully extracted {tables.n} tables.")

        # Combine all extracted tables (DataFrames) into a single DataFrame
        dfs = [table.df for table in tables]
        combined_df = pd.concat(dfs, ignore_index=True)



        # if list_of_raw_dataframes:
        #     # Get the raw combined DataFrame
        #     raw_df_with_headers = list_of_raw_dataframes[0].copy()
        
        #     # Apply preliminary cleaning to ensure headers are correctly set, as done in previous steps
        #     raw_df_with_headers.dropna(axis=1, how='all', inplace=True)
        #     raw_df_with_headers.dropna(axis=0, how='all', inplace=True)
            
        #     header_row_index = raw_df_with_headers[raw_df_with_headers.apply(lambda row: 'TIME' in str(row).upper(), axis=1)].index
            
        #     if len(header_row_index) > 0:
        #         header_index = header_row_index[0]
        #         # Drop rows before and including the header row
        #         final_df = raw_df_with_headers.iloc[header_index + 1:].copy()
        #     else:
        #         # If header not found, assume the first row is the first event
        #         final_df = raw_df_with_headers.copy()

        combined_df.dropna(axis=1, how='all', inplace=True)
        combined_df.dropna(axis=0, how='all', inplace=True)

        header_row_index = combined_df[combined_df.apply(lambda row: 'TIME' in str(row).upper(), axis=1)].index

        if len(header_row_index) > 0:
            header_index = header_row_index[0]
            # Drop rows before and including the header row
            final_df = combined_df.iloc[header_index + 1:].copy()
        else:
            # If header not found, assume the first row is the first event
            final_df = combined_df.copy()

        

        return final_df

    except Exception as e:
        print(f"An error occurred during PDF processing: {e}")
        return []

# --- PART 2: Data Normalization (The 'scrape_and_analyze_ncaa_game' Format) ---

def normalize_ncaa_pbp_dataframe(final_df, game, date, team, opponent):
    """
    Transforms the raw scraped DataFrame into a standardized NCAA play-by-play format.
    """
    df = final_df.copy()
    # print(df)

    # # 1. Clean Column Headers
    # # Rename columns based on the expected structure: TIME, UWW_ACTION, SCORE, EUR_ACTION
    # df.columns = ['TIME', 'UWW_ACTION', 'SCORE', 'EUR_ACTION']

    if len(df[df[1].str.contains('Whitewater')]) > 0:
        df.columns = ['TIME', 'TEAM_ACTION', 'SCORE', 'OPPONENT_ACTION']
        df[['TEAM_SCORE_FILLED', 'OPPONENT_SCORE_FILLED']] = df['SCORE'].str.split('-', expand=True)
        df['TEAM_ACTION'] = np.where(df['SCORE'].str.contains('Half'),df['SCORE'],df['TEAM_ACTION'])        
        df.drop(columns=['SCORE'], inplace=True)
    else:
        df.columns = ['TIME', 'OPPONENT_ACTION', 'SCORE', 'TEAM_ACTION']
        df[['OPPONENT_SCORE_FILLED', 'TEAM_SCORE_FILLED']] = df['SCORE'].str.split('-', expand=True)
        df.drop(columns=['SCORE'], inplace=True)

       

    
    # 2. Clean Data: Remove fully empty rows
    df.replace('', pd.NA, inplace=True)
    df.dropna(how='all', axis=0, subset=['TIME', 'TEAM_ACTION', 'OPPONENT_ACTION','TEAM_SCORE_FILLED','OPPONENT_SCORE_FILLED'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    df['Game'] = game
    df['Date'] = date
    df['TEAM_NAME'] = team
    df['OPPONENT_NAME'] = opponent
    
    

    # 3. Separate Time and Half
    # The TIME column is often in the format "MM:SS (HX)" or "Start 1st Half"
    
    # Initialize Half and Time_Remaining
    df['HALF'] = df['TIME'].apply(lambda x: re.search(r'\((H\d)\)', str(x)).group(1) if re.search(r'\((H\d)\)', str(x)) else ('H1' if '1st Half' in str(x) else ('H2' if '2nd Half' in str(x) else None)))
    df['HALF'] = df['HALF'].str.replace('H','').fillna(0).astype(int)
    df['TIME'] = df['TIME'].apply(lambda x: re.search(r'(\d{1,2}:\d{2})', str(x)).group(1) if re.search(r'(\d{1,2}:\d{2})', str(x)) else None)
    
    # Fill down the columns for events that span multiple rows (like Start/End Half)    
    df[['TIME','TEAM_SCORE_FILLED','OPPONENT_SCORE_FILLED','HALF']].ffill(inplace=True)    


    # Convert scores to numeric (coercing errors to NaN for non-score rows like '0-0')
    df['TEAM_SCORE_FILLED'] = pd.to_numeric(df['TEAM_SCORE_FILLED'], errors='coerce').astype('Int64')
    df['OPPONENT_SCORE_FILLED'] = pd.to_numeric(df['OPPONENT_SCORE_FILLED'], errors='coerce').astype('Int64')
    
    # Forward fill scores for continuity, but be careful not to apply to "Start" rows
    df['TEAM_SCORE_FILLED'].ffill(inplace=True)
    df['OPPONENT_SCORE_FILLED'].ffill(inplace=True)
    
    # Back-fill the initial "0-0" for the first entry
    df['TEAM_SCORE_FILLED'].bfill(limit=1, inplace=True)
    df['OPPONENT_SCORE_FILLED'].bfill(limit=1, inplace=True)

    # 5. Consolidate Actions
    # Create the final, normalized play-by-play columns

    df['ACTING_TEAM'] = np.where(df['TEAM_ACTION'].isna(),df['OPPONENT_NAME'],df['TEAM_NAME'])
    df['PLAY'] = np.where(df['TEAM_ACTION'].isna(),df['OPPONENT_ACTION'],df['TEAM_ACTION'])
    

    # 6. Final Cleanup and Selection
    # Select and reorder the final columns
    final_cols = ['Game', 'Date', 'HALF', 'TIME', 'TEAM_NAME','OPPONENT_NAME','TEAM_SCORE_FILLED', 'OPPONENT_SCORE_FILLED','ACTING_TEAM','PLAY']
    final_pbp_df = df[final_cols].copy()
    
    # # Drop rows where no action was recorded (e.g., intermediate rows cleaned up by the ffill)
    # final_pbp_df.dropna(subset=['Action'], inplace=True)
    
    # Reset index one last time
    final_pbp_df.reset_index(drop=True, inplace=True)
    
    # Replace any <NA> with a clean string representation for display
    final_pbp_df.fillna('', inplace=True)
    
    return final_pbp_df




# --- MAIN SCRAPING AND ANALYSIS FUNCTION (Refactored) ---

def scrape_and_analyze_ncaa_game(url, team_name, opponent_name, team_homeoraway,usepdf):
    """Orchestrates the scraping, cleaning, and analysis process."""
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    driver.get(url)
    df_pbp = pd.DataFrame() 
    original_box_score = None
         
        
            
    WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, f'//div[@class="{TABLE_CONTAINER_CLASS}"]')))
    html_source = driver.page_source
    tables = pd.read_html(io.StringIO(html_source))
    if len(tables) >= 2:
        print('found tables before button click')
        
        if team_homeoraway == 'away':
            team_box_score = tables[1]
            print("\n--- 🌐 Team Box Score Scraped from Initial Page (Table 2) ---")
            display(team_box_score)
            # teamroster = team_box_score['Name'].to_list()
            # to_be_removed = {'Total', np.nan}
            # teamroster = [item for item in teamroster if item not in to_be_removed ]
            opponent_box_score = tables[1]
            print("\n--- 🌐 Opponent Box Score Scraped from Initial Page (Table 2) ---")
            display(opponent_box_score)
        else:
            opponent_box_score = tables[1]
            print("\n--- 🌐 Opponent Box Score Scraped from Initial Page (Table 2) ---")
            display(opponent_box_score)
            # opponentroster = opponent_box_score['Name'].to_list()
            # to_be_removed = {'Total', np.nan}
            # opponentroster = [item for item in opponentroster if item not in to_be_removed ]
            team_box_score = tables[1]
            print("\n--- 🌐 Team Box Score Scraped from Initial Page (Table 2) ---")
            display(team_box_score)
                


    # The class name is used for a precise CSS selector.
    button_class = "boxscore-team-selector-team homeTeam-bg-primary_color awayTeam-border-primary_color home"
    button_css_selector = f'.{button_class.replace(" ", ".")}'

    # 2. Find and click the button
    print(f"2. Attempting to find and click the button...")
    
    # Use explicit wait to ensure the element is visible and clickable
    wait = WebDriverWait(driver, 20)
    team_button = wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, button_css_selector))
    )

    team_button.click()
    print("   Successfully clicked the team selector button.")

    # 3. Wait for content update
    # A short, explicit pause to allow the page's AJAX content to load after the click
    time.sleep(3) 
    print("3. Content update delay complete.")

    WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, f'//div[@class="{TABLE_CONTAINER_CLASS}"]')))
    html_source = driver.page_source
    tables = pd.read_html(io.StringIO(html_source))
    if len(tables) >= 2:
        if team_homeoraway == 'home':
            team_box_score = tables[1]
            print("\n--- 🌐 Team Box Score Scraped from Initial Page (Table 2) ---")
            display(team_box_score)
            # teamroster = team_box_score['Name'].to_list()
            # to_be_removed = {'Total', np.nan}
            # teamroster = [item for item in teamroster if item not in to_be_removed ]
            team_box_score = tables[1]
            print("\n--- 🌐 Team Box Score Scraped from Initial Page (Table 2) ---")
            display(team_box_score)
        else:
            opponent_box_score = tables[1]
            print("\n--- 🌐 Opponent Box Score Scraped from Initial Page (Table 2) ---")
            display(opponent_box_score)
            # opponentroster = opponent_box_score['Name'].to_list()
            # to_be_removed = {'Total', np.nan}
            # opponentroster = [item for item in opponentroster if item not in to_be_removed ]
            opponent_box_score = tables[1]
            print("\n--- 🌐 Opponent Box Score Scraped from Initial Page (Table 2) ---")
            display(opponent_box_score)

        
        
                
        # else:
        #     print("Warning: Could not find the original detailed box score table (expected at index 1).")

    print("Made it to drivers")

    if usepdf == 'no':
        # Navigate to the Play-by-Play page
        # WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, f'//div[@class="{TEAM_SELECTOR_CLASS}"]'))).click()
        # print("Made it passed driver 1")
        WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, f'//a[@href="{PLAY_BY_PLAY_HREF}"]'))).click()
        print("Made it passed driver 1")
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, f'//div[@class="{PLAY_BY_PLAY_TABLE_CONTAINER}"]')))
        print("Made it passed driver 2")
        time.sleep(2)
        
        print("Made it past drivers waiting")
    
        html_source = driver.page_source
        tables = pd.read_html(io.StringIO(html_source))
    
        print("Found Tables")
    
        if len(tables) >= 3:
            df_pbp = pd.concat(tables[1:3], ignore_index=True)
            df_pbp.columns = df_pbp.columns.str.upper()
        else:
            raise ValueError("Could not find enough tables in the Play-by-Play page source.")
    
        game_date = "Date not found"
    
        try:
            # Wait for the date element to be present within the gamecenterApp (optional, but good practice)
            date_element = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CLASS_NAME, "date"))
            )
            game_date = date_element.text.strip()
            print(f"\n--- 📅 Game Date Scraped ---")
            print(f"Date: {game_date}")
            
        except TimeoutException:
            print("\nWarning: Could not find the game date element within the timeout period.")
        except Exception as e:
            print(f"\nError scraping game date: {e}")
    
        driver.quit()
        
        # --- Phase 6: Add Game Details and Score and Half Calculation ---
        df_pbp['Game'] = GameNumber
        df_pbp['Date'] = pd.to_datetime(game_date)
        df_pbp['TEAM_NAME'] = team_name
        df_pbp['OPPONENT_NAME'] = opponent_name
        
            
        df_pbp['HALF'] = 1
    


        #Remove lines that have assist by same player as made basket
        df_pbp = df_pbp[(df_pbp['PLAY'].shift(1).str.split("'s",expand=True)[1]) != (np.where(df_pbp['PLAY'].str.contains('Assist by'),df_pbp['PLAY'].str.split("'s",expand=True)[1],np.nan))]
    
        if not df_pbp.empty:
            df_pbp.loc[0, 'SCORE'] = '(0-0)'
            
        
        
        score_components = df_pbp['SCORE'].str.replace(r'[\(\)]', '', regex=True).str.split('-', expand=True)
    
        if team_homeoraway == 'away':        
            df_pbp['TEAM_SCORE'] = pd.to_numeric(score_components[0], errors='coerce') 
            df_pbp['OPPONENT_SCORE'] = pd.to_numeric(score_components[1], errors='coerce')
        else:
            df_pbp['TEAM_SCORE'] = pd.to_numeric(score_components[1], errors='coerce') 
            df_pbp['OPPONENT_SCORE'] = pd.to_numeric(score_components[0], errors='coerce')
        
        df_pbp['OPPONENT_SCORE_FILLED'] = df_pbp['OPPONENT_SCORE'].ffill()
        df_pbp['TEAM_SCORE_FILLED'] = df_pbp['TEAM_SCORE'].ffill()
    
        df_pbp['ACTING_TEAM'] = np.where(df_pbp['PLAY'].str.contains(team_name),team_name,opponent_name)
        
        

    if 'yes' in usepdf:
        print('Play by Play unavailable, using pdf from fast scout')
        ########Include Eureka Game###########
        pdf_file_name = "Game_Data\\11_15_25 UW-Whitewater @ Eureka.pdf"
        date = '2025-11-15'
        
    
        final_df = scrape_pdf_to_dataframe(pdf_file_name,GameNumber,date,team_name,opponent_name)
        ncaa_pbp_df = normalize_ncaa_pbp_dataframe(final_df, GameNumber, date, team_name, opponent_name)
        df_pbp = pd.concat([df_pbp, ncaa_pbp_df])
        df_pbp.loc[0, 'HALF'] = 1
        df_pbp.loc[0, 'TIME'] = '20:00' 

       
        df_pbp['TIME'] = np.where(df_pbp['PLAY']=='End 1st Half','00:00',df_pbp['TIME'])
        
        df_pbp['TIME'] = np.where(df_pbp['PLAY']=='Start 2nd Half','20:00',df_pbp['TIME'])
        #######################################
    
        

    end_of_half_index = df_pbp[df_pbp['PLAY'] == "End of 1st Half."].index
    if not end_of_half_index.empty:
        df_pbp['HALF'] = np.where(df_pbp.index > end_of_half_index[0], 2 , df_pbp['HALF'])
        eoh_index = end_of_half_index[0]
    else:
        eoh_index = -1

    df_pbp['OPPONENT_POINTS'] = df_pbp['OPPONENT_SCORE_FILLED'].diff().fillna(0).astype(int)
    df_pbp['TEAM_POINTS'] = df_pbp['TEAM_SCORE_FILLED'].diff().fillna(0).astype(int)


    
    df_pbp['IS_END_OF_POSSESSION'] = df_pbp.apply(
        lambda row: determine_eop(row['PLAY'], row['TEAM_POINTS'], row['OPPONENT_POINTS']),
        axis=1
    )
    
    # --- Roster Tracking & Starter Determination (Refactored) ---

    # Substitution Strings (Normalized to lowercase 'in'/'out')
    TEAM_SUB_OUT_STRING_NEW = f"Subbing out for {team_name}-"
    TEAM_SUB_IN_STRING_NEW = f"Subbing in for {team_name}-"
    OPPONENT_SUB_OUT_STRING_NEW = f"Subbing out for {opponent_name}-"
    OPPONENT_SUB_IN_STRING_NEW = f"Subbing in for {opponent_name}-"

    if usepdf == 'no':
        teamroster = df_pbp[((df_pbp['PLAY'].str.contains(TEAM_SUB_IN_STRING_NEW)) | (df_pbp['PLAY'].str.contains(TEAM_SUB_OUT_STRING_NEW)))]['PLAY'].str.split('-').str[-1].unique()
        print('Team Roster:',teamroster)
        opponentroster = df_pbp[((df_pbp['PLAY'].str.contains(OPPONENT_SUB_IN_STRING_NEW)) | (df_pbp['PLAY'].str.contains(OPPONENT_SUB_OUT_STRING_NEW)))]['PLAY'].str.split('-').str[-1].unique()
        print('Opponent Roster:',opponentroster)
    else:
        teamroster = (df_pbp[((df_pbp['PLAY'].str.contains('Subs Out|Subs In')) & (df_pbp['ACTING_TEAM']==team_name))]['PLAY'].str.split(' ').str[0] + ' ' + 
            df_pbp[((df_pbp['PLAY'].str.contains('Subs Out|Subs In')) & (df_pbp['ACTING_TEAM']==team_name))]['PLAY'].str.split(' ').str[1]).unique()
        print('Team Roster:',teamroster)
        opponentroster = (df_pbp[((df_pbp['PLAY'].str.contains('Subs Out|Subs In')) & (df_pbp['ACTING_TEAM']==opponent_name))]['PLAY'].str.split(' ').str[0] + ' ' + 
            df_pbp[((df_pbp['PLAY'].str.contains('Subs Out|Subs In')) & (df_pbp['ACTING_TEAM']==opponent_name))]['PLAY'].str.split(' ').str[1]).unique()
        print('Opponent Roster:',opponentroster)

    # Initialize unified stat columns (STL added here)
    find_team = '(' + '|'.join(re.escape(name) for name in teamroster) + ')'
    find_opponent = '(' + '|'.join(re.escape(name) for name in opponentroster) + ')'
    
    df_pbp['PLAYER'] = np.where(df_pbp['ACTING_TEAM']==team_name, 
                            df_pbp['PLAY'].str.extract(find_team, expand=False),
                            df_pbp['PLAY'].str.extract(find_opponent, expand=False))


    
    
    
    # Get starters using the new unified function
    TEAM_start_list = get_starters_list(df_pbp, TEAM_SUB_OUT_STRING_NEW, TEAM_SUB_IN_STRING_NEW, usepdf, team_name)
    OPPONENT_start_list = get_starters_list(df_pbp, OPPONENT_SUB_OUT_STRING_NEW, OPPONENT_SUB_IN_STRING_NEW, usepdf, opponent_name)



    # Lineup Tracking (Refactored to single loop)
    roster_data = {
        'TEAM': {
            'roster_set': set(TEAM_start_list),
            'sub_out_str': TEAM_SUB_OUT_STRING_NEW,
            'sub_in_str': TEAM_SUB_IN_STRING_NEW,
            'roster_list': []
        },
        'OPPONENT': {
            'roster_set': set(OPPONENT_start_list),
            'sub_out_str': OPPONENT_SUB_OUT_STRING_NEW,
            'sub_in_str': OPPONENT_SUB_IN_STRING_NEW,
            'roster_list': []
        }
    }

    # NEW: Dictionary to track the player who subbed out, waiting for the sub-in player.
    # Key: 'TEAM' or 'OPPONENT', Value: Player Name (or None)
    pending_sub_out = {'TEAM': None, 'OPPONENT': None}


    for _, row in df_pbp.iterrows():
    
        play = row['PLAY']
        actionby = row['ACTING_TEAM']
        player_name = row['PLAYER']
    
           
        for team_key in roster_data.keys():
            # print(team_key)
    
            team_data = roster_data[team_key]
            
            # Determine the team name for the helper function
            current_team_name = team_name if team_key == 'TEAM' else opponent_name
            
            if ('Subbing out' in play or 'Subs Out' in play) and current_team_name in actionby:
                # print(play)
                # print('Before:',team_data['roster_set'])
                try:
                    team_data['roster_set'].remove(player_name)
                    pending_sub_out[team_key] = player_name
                    # print('After:',team_data['roster_set'])
                except Exception as e:
                    print(f"\nPlayer not in roster on court: {e}")
    
            if ('Subbing in' in play or 'Subs In' in play) and current_team_name in actionby:
                # print(play)
                # print('Before:',team_data['roster_set'])
                try:
                    team_data['roster_set'].add(player_name)
                    pending_sub_out[team_key] = None
                    # print('After:',team_data['roster_set'])
                except Exception as e:
                    print(f"\nPlayer not in roster on court: {e}")
            
            # if action == 'OUT':
            #     # Player is subbing out. Remove from court and store as pending.
            #     if player_name in team_data['roster_set']:
            #         team_data['roster_set'].remove(player_name)
            #         pending_sub_out[team_key] = player_name # Store the player that just left
            #     # else: Player wasn't on the court (data error, skip)
            
            # elif action == 'IN':
            #     # Player is subbing in. Add to court and clear the pending state.
            #     # This completes the substitution event.
            #     if player_name not in team_data['roster_set']:
            #         team_data['roster_set'].add(player_name)
            #         # Clear the pending player
            #         pending_sub_out[team_key] = None 
            #     # else: Player was already on the court (data error, skip)

        # --- Record Current Roster State ---

        # print(team_data)
        # Record the roster state for the current row (after any potential change)
        for team_data in roster_data.values():
            team_data['roster_list'].append(",".join(sorted(team_data['roster_set'])))
    
    df_pbp['TEAM_ROSTER_ON_COURT'] = roster_data['TEAM']['roster_list']
    df_pbp['OPPONENT_ROSTER_ON_COURT'] = roster_data['OPPONENT']['roster_list']


    df_pbp['TIME_SECONDS'] = df_pbp['TIME'].apply(time_to_seconds)
    df_pbp['NEXT_TIME_SECONDS'] = df_pbp['TIME_SECONDS'].shift(-1)
    
    if eoh_index != -1:
        df_pbp.loc[eoh_index, 'NEXT_TIME_SECONDS'] = 0.0

    df_pbp['NEXT_TIME_SECONDS'] = np.where(df_pbp['PLAY']=='End 1st Half',0.0,df_pbp['NEXT_TIME_SECONDS'])
    
    # Ensure correct duration calculation (time always flows from high to low seconds)
    df_pbp['DURATION_SECONDS'] = df_pbp['TIME_SECONDS'] - df_pbp['NEXT_TIME_SECONDS']
    df_pbp['DURATION_SECONDS'] = df_pbp['DURATION_SECONDS'].fillna(df_pbp['TIME_SECONDS']).abs()
    
    # --- Phase 7: ENRICH PBP DF WITH UNIFIED STATS AND ACTING_TEAM ---
    df_pbp_enriched = add_player_stats_to_pbp(df_pbp, team_name, opponent_name, usepdf)

    # Make sure OPPONENT and TEAM Rosters aren't null
    df_pbp_enriched['OPPONENT_ROSTER_ON_COURT'] = np.where(df_pbp_enriched['OPPONENT_ROSTER_ON_COURT'].isna(),'None',df_pbp_enriched['OPPONENT_ROSTER_ON_COURT'])
    df_pbp_enriched['TEAM_ROSTER_ON_COURT'] = np.where(df_pbp_enriched['TEAM_ROSTER_ON_COURT'].isna(),'None',df_pbp_enriched['TEAM_ROSTER_ON_COURT'])

    # --- Phase 8: Add information from scouting reports ---

    scouting = pd.read_csv('Game_Data\\Scouting.csv')
    scouting = scouting[scouting['PART OF REPORT']=='Player Notes']

    df_pbp_enriched = add_roster_traits_to_dataframe(df_pbp_enriched, scouting, 'OPPONENT_ROSTER_ON_COURT')
    print('Finished adding scouting report data')

    
    # --- Phase 9: Create Box Scores using Aggregation on Enriched PBP ---
    team_box_score_pbp = create_box_score(df_pbp_enriched, team_name)
    opponent_box_score_pbp = create_box_score(df_pbp_enriched, opponent_name)
    
    
    

    agg_dict = {
        'DURATION_SECONDS': 'sum',
        'TEAM_POINTS': 'sum',
        'OPPONENT_POINTS': 'sum',
        'IS_END_OF_POSSESSION': 'sum'
    }

    # Use the unified function for both teams
    TEAM_final_results = calculate_lineup_ratings(df_pbp, team_name, opponent_name, agg_dict)
    OPPONENT_final_results = calculate_lineup_ratings(df_pbp, opponent_name, team_name, agg_dict)
    
    return {
        'html_source': html_source,
        'df_pbp': df_pbp,
        'pbp_data': df_pbp_enriched, 
        'TEAM_lineup_analysis': TEAM_final_results,
        'OPPONENT_lineup_analysis': OPPONENT_final_results,
        'team_box_score_pbp': team_box_score_pbp,
        'opponent_box_score_pbp': opponent_box_score_pbp,
        'team_box_score': team_box_score,
        'opponent_box_score': opponent_box_score,
        'teamroster': teamroster,
        'opponentroster': opponentroster
    }

# --- BOX SCORE COMPARISON AND CORRECTION FUNCTION ---

def compare_box_scores(original_df, derived_df, df_pbp_enriched, team_name):
    """
    Compares the original and derived 3PA, calculates the difference, and corrects 
    the underlying df_pbp_enriched for players with Derived > Original by resetting
    'Jumper MISSED' 3PA values to 0 while retaining FGA=1.
    """
    print(f"\n## 🔍 Box Score Comparison & Correction: {team_name} (3-Point Attempts Validation) 🔍")
    print("-----------------------------------------------------------------------------------")
    
    # 1. Prepare Original Scraped Data for Comparison
    player_rows_mask = original_df.iloc[:, 1].notna()
    original_players = original_df[player_rows_mask].copy()

    # Column names used for comparison
    new_columns = ['Jersey/Team', 'Player', 'POS', 'MIN', 'FG', '3FG', 'FT', 'REB_OT', 'TOT', 'AST', 'PF', 'STL', 'TO', 'BLK', 'PTS']
    original_players.columns = new_columns + original_players.columns.tolist()[len(new_columns):]
    
    original_players = original_players[
        ~original_players['Player'].astype(str).str.contains('TOTAL|TEAM|DNP', na=False)
    ].copy()
        
    original_players['3FG_str'] = original_players['3FG'].astype(str).str.replace('DNP|dnp', '0-0', regex=True)
    original_players['3PA_orig'] = original_players['3FG_str'].str.split('-', expand=True)[1].astype(int)
    original_comp = original_players.set_index('Player')[['3PA_orig']].sort_index()
    
    # 2. Prepare Derived PBP Data for Comparison
    derived_comp = derived_df[derived_df['PLAYER'] != 'TOTAL'].copy()
    derived_comp = derived_comp.set_index('PLAYER')[['3PA']].sort_index()

    # 3. Perform Comparison and Correction
    
    # Merge the two tables on the player index
    comparison_df = original_comp.merge(derived_comp, left_index=True, right_index=True, how='inner')
    comparison_df.columns = ['Original 3PA', 'Derived 3PA']
    
    # Calculate the difference column
    comparison_df['Difference'] = comparison_df['Derived 3PA'] - comparison_df['Original 3PA']
    
    # Initialize a list to hold players that needed correction
    players_corrected = []

    # Iterate over players to perform corrections
    for player, row in comparison_df.iterrows():
        diff = row['Difference']
        
        # Only correct if the Derived count is too high
        if diff > 0:
            print(f"\nCorrection Needed for **{player}**: Derived 3PA is {diff} higher than Original.")
            
            # Identify the excess 'Jumper MISSED' plays for this player
            correction_indices = df_pbp_enriched[
                (df_pbp_enriched['PLAYER'] == player) & 
                (df_pbp_enriched['ACTING_TEAM'] == team_name) & 
                (df_pbp_enriched['PLAY'].str.contains('Jumper MISSED', na=False)) & 
                (df_pbp_enriched['3PA'] == 1) 
            ].head(diff).index 

            if not correction_indices.empty:
                # Set 3PA to 0, but KEEP FGA at 1 (as it's still a field goal attempt)
                df_pbp_enriched.loc[correction_indices, '3PA'] = 0
                df_pbp_enriched.loc[correction_indices, 'FGA'] = 1 
                
                print(f"  -> Corrected **{len(correction_indices)}** 'Jumper MISSED' plays in PBP data (3PA set to 0, FGA kept at 1).")
                players_corrected.append(player)
            else:
                print(f"  -> WARNING: Could not find {diff} 'Jumper MISSED' plays to correct for {player}.")

    # 4. Recalculate Box Score after Correction
    if players_corrected:
        print("\n### Recalculating Box Score after PBP Correction...")
        derived_df_corrected = create_box_score(df_pbp_enriched, team_name)
        
        # Get corrected 3PA and 3P% for final comparison display
        derived_comp_corrected = derived_df_corrected[derived_df_corrected['PLAYER'] != 'TOTAL'].set_index('PLAYER')[['3PA', '3P%']].sort_index()
        
        # Merge the original comparison with the new corrected derived data
        final_comparison_df = original_comp.merge(derived_comp_corrected, left_index=True, right_index=True, how='inner')
        final_comparison_df.columns = ['Original 3PA', 'Corrected Derived 3PA', 'Corrected Derived 3P%']
        final_comparison_df['Final Difference'] = final_comparison_df['Corrected Derived 3PA'] - final_comparison_df['Original 3PA']
        final_comparison_df['Final Status'] = final_comparison_df['Final Difference'].apply(lambda x: '✅ Match' if x == 0 else '❌ MISMATCH')
        
        comparison_to_display = final_comparison_df[['Original 3PA', 'Corrected Derived 3PA', 'Final Difference', 'Final Status', 'Corrected Derived 3P%']]
        
        # Recalculate totals
        total_original = comparison_to_display['Original 3PA'].sum()
        total_derived_corrected = final_comparison_df['Corrected Derived 3PA'].sum()
        total_status = '✅ Match' if total_original == total_derived_corrected else '❌ MISMATCH'

        print("\n### Player-by-Player 3PT Attempts (POST-CORRECTION)")
        display(comparison_to_display)
        
        print(f"\n### Team Total Comparison (POST-CORRECTION)")
        print(f"* **Original Total 3PA:** {total_original}")
        print(f"* **Corrected Derived Total 3PA:** {total_derived_corrected}")
        print(f"* **Total Status:** {total_status}")
    else:
        # If no corrections were needed, just print the initial comparison
        comparison_df['Status'] = comparison_df['Difference'].apply(lambda x: '✅ Match' if x == 0 else '❌ MISMATCH')
        comparison_to_display = comparison_df[['Original 3PA', 'Derived 3PA', 'Difference', 'Status']]
        
        total_original = comparison_df['Original 3PA'].sum()
        total_derived = comparison_df['Derived 3PA'].sum()
        total_status = '✅ Match' if total_original == total_derived else '❌ MISMATCH'
        
        print("\n### Player-by-Player 3PT Attempts (No Correction Applied)")
        display(comparison_to_display)
        
        print(f"\n### Team Total Comparison")
        print(f"* **Original Total 3PA:** {total_original}")
        print(f"* **Derived Total 3PA:** {total_derived}")
        print(f"* **Total Status:** {total_status}")
        derived_df_corrected = derived_df # Use original derived if no correction was made
        
    print("\n*Note: A positive 'Difference' meant the Derived PBP value was higher, and corrections were applied in the PBP data.*")
    
    # Use the corrected PBP data for the rest of the flow
    return df_pbp_enriched

# --- EXECUTION ---
if __name__ == "__main__":
    results = scrape_and_analyze_ncaa_game(URL, team_name, opponent_name, team_homeoraway, usepdf)
    
       
    if results:
        print("\n✅ Execution Complete. All scraping and initial statistical analysis phases finished.")
        
        # --- PHASE 10: COMPARISON AND CORRECTION (TEAM TEAM) ---
        corrected_pbp_data = compare_box_scores(
            results['team_box_score'], 
            results['team_box_score_pbp'], 
            results['pbp_data'],
            team_name
        )

        
        
        # --- PHASE 11: RECALCULATE FINAL BOX SCORE AFTER CORRECTION (for final output) ---
        final_TEAM_box_score = create_box_score(corrected_pbp_data, team_name)
        
        print(f"\n## 🏆 Final Derived Box Score: {team_name} (Post-Correction) 🏆")
        print("----------------------------------------------------------------------")
        display(final_TEAM_box_score)

        corrected_pbp_data.to_csv('Game_Data\\'+GameNumber+'.csv', index=False)
        print(f"\n## 🏆 Corrected PBP Data Saved 🏆")

        


found tables before button click

--- 🌐 Opponent Box Score Scraped from Initial Page (Table 2) ---


,NO,Name,POS,MIN,FGM-A,3PM-A,FTM-A,OREB,REB,AST,ST,BLK,TO,PF,PTS,Unnamed: 15
0,15,Vinnie Adjahoungbeta,F,23.7,1-5,0-3,3-4,0.0,6.0,0.0,0.0,2.0,3.0,2.0,5.0,NaN
1,NaN,Sebastian Blachut,G,35.5,5-8,0-2,4-6,1.0,2.0,4.0,0.0,0.0,4.0,2.0,14.0,NaN
2,40,Aidyn Boone,F,20.4,4-6,2-4,2-2,1.0,2.0,0.0,1.0,1.0,1.0,3.0,12.0,NaN
3,11,Jack Cherry,G,24.4,2-3,0-0,3-4,0.0,6.0,3.0,2.0,0.0,0.0,4.0,7.0,NaN
4,4,EJ Marshall,G,17.0,4-6,2-3,0-0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,10.0,NaN
5,21,Talen Pearson,G,13.6,1-2,1-2,0-0,0.0,1.0,0.0,0.0,0.0,2.0,2.0,3.0,NaN
6,2,Luke Smith,NaN,19.3,0-4,0-1,1-2,0.0,1.0,4.0,0.0,0.0,2.0,1.0,1.0,NaN
7,1,Dominic Trelenberg,G,37.9,7-17,2-6,2-2,3.0,5.0,2.0,0.0,0.0,0.0,1.0,18.0,NaN
8,22,Jeffrey Williams,F,8.4,1-2,0-1,0-1,1.0,2.0,0.0,1.0,0.0,0.0,0.0,2.0,NaN
9,Total,Total,Total,Total,25-53,7-22,15-21,8.0,30.0,13.0,4.0,3.0,12.0,16.0,72.0,NaN



--- 🌐 Team Box Score Scraped from Initial Page (Table 2) ---


,NO,Name,POS,MIN,FGM-A,3PM-A,FTM-A,OREB,REB,AST,ST,BLK,TO,PF,PTS,Unnamed: 15
0,15,Vinnie Adjahoungbeta,F,23.7,1-5,0-3,3-4,0.0,6.0,0.0,0.0,2.0,3.0,2.0,5.0,NaN
1,NaN,Sebastian Blachut,G,35.5,5-8,0-2,4-6,1.0,2.0,4.0,0.0,0.0,4.0,2.0,14.0,NaN
2,40,Aidyn Boone,F,20.4,4-6,2-4,2-2,1.0,2.0,0.0,1.0,1.0,1.0,3.0,12.0,NaN
3,11,Jack Cherry,G,24.4,2-3,0-0,3-4,0.0,6.0,3.0,2.0,0.0,0.0,4.0,7.0,NaN
4,4,EJ Marshall,G,17.0,4-6,2-3,0-0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,10.0,NaN
5,21,Talen Pearson,G,13.6,1-2,1-2,0-0,0.0,1.0,0.0,0.0,0.0,2.0,2.0,3.0,NaN
6,2,Luke Smith,NaN,19.3,0-4,0-1,1-2,0.0,1.0,4.0,0.0,0.0,2.0,1.0,1.0,NaN
7,1,Dominic Trelenberg,G,37.9,7-17,2-6,2-2,3.0,5.0,2.0,0.0,0.0,0.0,1.0,18.0,NaN
8,22,Jeffrey Williams,F,8.4,1-2,0-1,0-1,1.0,2.0,0.0,1.0,0.0,0.0,0.0,2.0,NaN
9,Total,Total,Total,Total,25-53,7-22,15-21,8.0,30.0,13.0,4.0,3.0,12.0,16.0,72.0,NaN


2. Attempting to find and click the button...
   Successfully clicked the team selector button.
3. Content update delay complete.

--- 🌐 Team Box Score Scraped from Initial Page (Table 2) ---


...

Made it to drivers
Made it passed driver 1
Made it passed driver 2
Made it past drivers waiting
Found Tables

--- 📅 Game Date Scraped ---
Date: DECEMBER 2ND 2025
Team Roster: ['Rashad Rogers' 'Brock Marino' 'Richie Warren' 'Jake Quast'
 'Isaac Verges' 'Joey Berezowitz' 'Luke Bara' 'Darius Chestnut'
 'Collin Madson' 'JR Lukenbill' 'Kelton McEwen']
Opponent Roster: ['Vinnie Adjahoungbeta' 'Talen Pearson' 'Luke Smith' 'Jack Cherry'
 'Aidyn Boone' 'Jeffrey Williams' 'Sebastian Blachut' 'Dominic Trelenberg'
 'EJ Marshall']
Not using PDF
Not using PDF
Finished adding scouting report data

✅ Execution Complete. All scraping and initial statistical analysis phases finished.

## 🔍 Box Score Comparison & Correction: Wis.-Whitewater (3-Point Attempts Validation) 🔍
-----------------------------------------------------------------------------------

Correction Needed for **Luke Bara**: Derived 3PA is 1 higher than Original.
  -> Corrected **1** 'Jumper MISSED' plays in PBP data (3PA set to 0, FGA k

,Original 3PA,Corrected Derived 3PA,Final Difference,Final Status,Corrected Derived 3P%
Brock Marino,0,0,0,✅ Match,0.00
Collin Madson,2,2,0,✅ Match,0.50
Darius Chestnut,0,0,0,✅ Match,0.00
Isaac Verges,2,2,0,✅ Match,0.00
JR Lukenbill,2,2,0,✅ Match,0.00
Jake Quast,1,1,0,✅ Match,0.00
Joey Berezowitz,1,1,0,✅ Match,0.00
Kelton McEwen,2,2,0,✅ Match,0.50
Luke Bara,8,8,0,✅ Match,0.75
Richie Warren,1,1,0,✅ Match,0.00



### Team Total Comparison (POST-CORRECTION)
* **Original Total 3PA:** 19
* **Corrected Derived Total 3PA:** 19
* **Total Status:** ✅ Match

*Note: A positive 'Difference' meant the Derived PBP value was higher, and corrections were applied in the PBP data.*

## 🏆 Final Derived Box Score: Wis.-Whitewater (Post-Correction) 🏆
----------------------------------------------------------------------


,PLAYER,PTS,FGM,FGA,FG%,3PM,3PA,3P%,FTM,FTA,FT%,OREB,DREB,REB,AST,STL,TO,FOULS
0,Brock Marino,2,1,1,1.000,0,0,0.000,0,2,0.00,0,1,1,0,0,0,0
1,Collin Madson,16,6,8,0.750,1,2,0.500,3,3,1.00,2,6,8,1,1,0,2
2,Darius Chestnut,0,0,0,0.000,0,0,0.000,0,0,0.00,0,0,0,1,0,0,2
3,Isaac Verges,13,5,7,0.714,0,2,0.000,3,4,0.75,1,4,5,3,1,0,0
4,JR Lukenbill,7,3,5,0.600,0,2,0.000,1,1,1.00,0,2,2,1,0,0,1
5,Jake Quast,9,4,5,0.800,0,1,0.000,1,2,0.50,0,1,1,2,2,1,2
6,Joey Berezowitz,2,1,2,0.500,0,1,0.000,0,0,0.00,0,1,1,1,1,0,2
7,Kelton McEwen,3,1,2,0.500,1,2,0.500,0,0,0.00,0,0,0,0,0,0,1
8,Luke Bara,24,8,11,0.727,6,8,0.750,2,2,1.00,0,1,1,2,0,0,5
9,Rashad Rogers,2,1,2,0.500,0,0,0.000,0,0,0.00,0,1,1,0,0,1,2



## 🏆 Corrected PBP Data Saved 🏆
